In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/local/opt/openjdk@17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/01 21:31:16 WARN Utils: Your hostname, Jaimes-MacBook-Pro-3.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.10 instead (on interface en0)
26/03/01 21:31:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 21:31:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_green = spark.read.option("recursiveFileLookup", "true") \
    .parquet("./data/pq/green/")

In [4]:
df_green.registerTempTable('green')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [5]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [6]:
df_green_revenue.show()

[Stage 1:===================================================>       (7 + 1) / 8]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 03:00:00| 181|315.61000000000007|            14|
|2020-01-05 01:00:00|  42|163.10000000000005|            14|
|2020-01-06 13:00:00|  22|            337.62|             6|
|2020-01-21 14:00:00|  33|            571.46|            29|
|2020-01-06 14:00:00| 122|            110.26|             2|
|2020-01-19 17:00:00|  41| 488.8400000000001|            38|
|2020-01-08 09:00:00| 210|184.07999999999998|            10|
|2020-01-23 21:00:00| 129| 84.77999999999999|             9|
|2020-01-12 22:00:00| 181|284.01000000000005|            15|
|2020-01-05 16:00:00|  97|498.45000000000016|            39|
|2020-01-23 14:00:00| 185|             31.75|             2|
|2020-01-20 13:00:00| 166| 318.9400000000001|            24|
|2020-01-31 23:00:00|  76| 67.74000000000001|             2|
|2020-01-13 23:00:00| 22

In [7]:
df_green_revenue \
.repartition(20) \
.write.parquet('data/report/revenue/green', mode='overwrite')

26/03/01 21:34:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/01 21:34:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [8]:
df_yellow = spark.read.option("recursiveFileLookup", "true") \
    .parquet("./data/pq/yellow/")

df_yellow.registerTempTable('yellow')

In [9]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [10]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

26/03/01 21:36:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/01 21:36:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/01 21:36:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [11]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [12]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [13]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [14]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

26/03/01 21:37:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [15]:
df_join = spark.read.parquet('data/report/revenue/total')

In [16]:
df_join.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|   7| 769.7299999999997|                  45| 455.1700000000001|                   38|
|2020-01-01 00:00:00|  10|              NULL|                NULL|             42.41|                    2|
|2020-01-01 00:00:00|  34|              NULL|                NULL|              19.3|                    1|
|2020-01-01 00:00:00|  36|295.34000000000003|                  11|            109.17|                    3|
|2020-01-01 00:00:00|  37|175.67000000000002|                   6|            161.61|                    7|
|2020-01-01 00:00:00|  38| 98.78999999999999|                   2|              NULL|                 NULL|
|2020-01-01 00:00:00|  43|10

In [19]:
df_zone = spark.read \
    .option("header", "true") \
    .csv('data/taxi_zone_lookup.csv')

In [20]:
df_zone.show()

+----------+-------------+--------------------+------------+
|locationid|      borough|                zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [23]:
df_zone.write.parquet('data/zones')

In [25]:
df_zones = spark.read.parquet('data/zones/')

In [27]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.locationid)

In [28]:
df_result.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|locationid|  borough|                zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-01-01 00:00:00|   7| 769.7299999999997|                  45| 455.1700000000001|                   38|         7|   Queens|             Astoria|   Boro Zone|
|2020-01-01 00:00:00|  10|              NULL|                NULL|             42.41|                    2|        10|   Queens|        Baisley Park|   Boro Zone|
|2020-01-01 00:00:00|  34|              NULL|                NULL|              19.3|                    1|        34| Brooklyn|  Brooklyn Navy Yard|   Boro Zone|
|2020-01-01 00:00:00| 

In [30]:
df_result.drop('locationid', 'zone').write.parquet('data/revenue-zones')

26/03/01 21:46:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [31]:
spark.stop()